In [1]:
# Importowanie najwazniejszych komponentow Sionna RT
from sionna.rt import PathSolver, subcarrier_frequencies
from sionna_env import setup_scene, update_object_position
import ofdm_config as cfg
from IPython.display import clear_output
import numpy as np
import matplotlib.pyplot as plt

scene_file = "./OpenStreetMap/kampus_AGH_test_all/Kampus_AGH_B9_test_all.xml"

scene, tx, rx = setup_scene(scene_file, cfg.fc, cfg.rx_rows, cfg.rx_cols)

dron_target = scene.get("dron_target")
# print(f'Pozycja drona: {dron_target.position}')

samochod_target_b = scene.get("samochod_target_b")
# print(f'Pozycja samochodu_b: {samochod_target_b.position}')

samochod_target_d = scene.get("samochod_target_d")
# print(f'Pozycja samochodu_d: {samochod_target_d.position}')

czlowiek_target = scene.get("czlowiek_target")
# print(f'Pozycja czlowieka: {czlowiek_target.position}')

print(f'Pomyślnie załadowano scenę: {scene_file}')

Pomyślnie załadowano scenę: ./OpenStreetMap/kampus_AGH_test_all/Kampus_AGH_B9_test_all.xml


In [2]:
p_solver = PathSolver()

save_size = 100 # Liczba próbek do zapisania w pliku .npz
h_cfr_buffor = [] # Bufor na próbki kanału CFR
num_steps = 200 # Liczba kroków symulacji
frame_time = 0.1  # Czas trwania jednej klatki w sekundach

frequencies = subcarrier_frequencies(num_subcarriers=cfg.num_subcarriers, subcarrier_spacing=cfg.subcarrier_spacing)

# pętla symulacyjna
for i in range(num_steps):
    clear_output(wait=True)
    print(f"Obliczanie kroku {i+1}/{num_steps}")

    # Ruch samochodu B
    if i == 195:
        samochod_target_b.position = [0.0, 0.0, -50.0]
        samochod_target_b_vel = np.array([0, 0, 0], dtype=float)
    elif i < 195:
        if i > 120:
            samochod_target_b_vel = np.array([11, 0, 0], dtype=float)
        else:
            samochod_target_b_vel = np.array([9, -0.2, 0], dtype=float)

        update_object_position(samochod_target_b, samochod_target_b_vel, frame_time)

    # Ruch samochodu D
    if i <= 190:
        if i > 180:
            samochod_target_d_vel = np.array([-6.4, 0, 0], dtype=float)
        elif i > 170:
            samochod_target_d_vel = np.array([-5, 0, 0], dtype=float)
        elif i > 80:
            samochod_target_d_vel = np.array([0, 8.8, 0], dtype=float)
        elif i > 50:
            samochod_target_d_vel = np.array([14, 0, 0], dtype=float)
        else:
            samochod_target_d_vel = np.array([16, 0, 0], dtype=float)

        update_object_position(samochod_target_d, samochod_target_d_vel, frame_time)

    # Ruch człowieka
    if i > 150:
        czlowiek_target_vel = np.array([0, 1.5, 0], dtype=float)
    elif i > 100:
        czlowiek_target_vel = np.array([-1, 1.3, 0], dtype=float)
    else:
        czlowiek_target_vel = np.array([0, 1.3, 0], dtype=float)

    update_object_position(czlowiek_target, czlowiek_target_vel, frame_time)

    # Obliczanie ścieżek propagacji
    paths = p_solver(scene=scene, max_depth=3, samples_per_src=1_000_000, diffuse_reflection=True, specular_reflection=True, seed=67)

    freq_chunk_size = 8
    h_cfr_chunks = []

    for f_start in range(0, len(frequencies), freq_chunk_size):
        f_end = min(f_start + freq_chunk_size, len(frequencies))
        freq_chunk = frequencies[f_start:f_end]

        h_chunk = paths.cfr(frequencies=freq_chunk,
                          sampling_frequency=1 / cfg.total_symbol_duration,
                          num_time_steps=cfg.num_symbols,
                          normalize_delays=False,
                          normalize=False,
                          out_type="numpy")
        
        h_cfr_chunks.append(h_chunk)

    h_cfr = np.concatenate(h_cfr_chunks, axis=-1)

    h_cfr_buffor.append(h_cfr)

    if len(h_cfr_buffor) == save_size or i == num_steps - 1:

        chunk_data = np.stack(h_cfr_buffor, axis=0).astype(np.complex64)

        filename = f"simulation_data_combined_{(i//save_size) + 1}.npz"

        np.savez_compressed(filename, h_cfr=chunk_data)

        print(f"Zapisano próbki kanału CFR do pliku: {filename}")

        h_cfr_buffor = []

print("Symulacja zakończona.")

Obliczanie kroku 200/200
Zapisano próbki kanału CFR do pliku: simulation_data_combined_2.npz
Symulacja zakończona.
